# TP 2 · Analyser le journal brut des bornes

Question d'exploitation : **quelles bornes tombent le plus en panne ?**
On y répond avec l'API RDD uniquement. L'énoncé complet, avec les résultats
attendus, est dans `enonce.md`.

## Étape 0 · Démarrer une session

Cette cellule est fournie : elle crée la session et calcule les chemins.

In [ ]:
import os
import re
import shutil
import time

from pyspark.sql import SparkSession

try:
    ICI = os.path.dirname(os.path.abspath(__file__))
except NameError:  # dans un notebook, __file__ n'existe pas
    ICI = os.getcwd()
TP = os.path.dirname(ICI)
JOURNAL = os.path.join(TP, "donnees", "journal-bornes.log")
STATIONS = os.path.join(TP, "donnees", "stations.csv")
SORTIES = os.path.join(TP, "sorties")

spark = (SparkSession.builder
         .master("local[2]")
         .appName("tp2-journal-bornes")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(sc.version, sc.master, sc.defaultParallelism)

## Étape 1 · Charger le journal et le compter

Lisez le journal avec `sc.textFile`, affichez le nombre de partitions et de
lignes, puis un aperçu de trois lignes.

Sous Windows, `take()` sur le RDD brut peut faire planter le worker Python :
utilisez `spark.read.text(JOURNAL).show(3, truncate=False)` pour l'aperçu.

In [ ]:
# TODO : lignes = ...
# TODO : afficher getNumPartitions() et count()
# TODO : afficher un aperçu

## Étape 2 · Analyser chaque ligne, compter les invalides

`analyser(ligne)` renvoie `[(horodatage, station, type, champs)]` si la ligne
est valide, `[]` sinon. Les lignes écartées incrémentent l'accumulateur
`invalides`.

Indice : `ligne.split(" ", 3)`, puis `PAIRE.findall(reste)`.

In [ ]:
TYPES = {"RETRAIT": {"velo", "badge"},
         "DEPOT": {"velo", "duree"},
         "ERREUR": {"code", "message"}}
PAIRE = re.compile(r'(\w+)=("[^"]*"|\S+)')
STATION = re.compile(r"^S\d{3}$")

invalides = sc.accumulator(0)


def analyser(ligne):
    # TODO : découper, valider, construire le dictionnaire des champs
    return []


# TODO : evenements = lignes.flatMap(analyser)
# TODO : afficher evenements.count() et invalides.value

Appelez une seconde action sur `evenements`. Que vaut maintenant
`invalides.value` ? Pourquoi ?

In [ ]:
# TODO

## Étape 3 · Mettre en cache, et mesurer

Mesurez un `count()` sans cache, puis deux `count()` successifs sur le RDD en
cache. Remettez l'accumulateur à zéro avant.

In [ ]:
def chrono(action):
    debut = time.perf_counter()
    action()
    return time.perf_counter() - debut


# TODO

## Étape 4 · Événements par type

In [ ]:
# TODO : map vers (type, 1), puis reduceByKey

## Étape 5 · Les dix stations les plus en panne

Chargez `stations.csv` en dictionnaire `{station_id: nom}`, diffusez-le avec
`sc.broadcast`, puis comptez les erreurs par station.

In [ ]:
# TODO : noms = ...
# TODO : noms_diffuses = sc.broadcast(noms)
# TODO : top_pannes = ...

## Étape 6 · Répartition des codes d'erreur

In [ ]:
# TODO

## Étape 7 · Batterie moyenne au retrait des vélos électriques

La moyenne n'est pas associative : réduisez des couples `(somme, effectif)`.

In [ ]:
# TODO

## Étape 8 · Écrire les résultats

Une ligne `identifiant;nom;nombre` par station, en un seul fichier, dans
`sorties/tp2-top-pannes`.

In [ ]:
cible = os.path.join(SORTIES, "tp2-top-pannes")
shutil.rmtree(cible, ignore_errors=True)   # saveAsTextFile refuse d'écraser
# TODO

## Étape 9 · Soumettre le script

Exportez ce notebook en script, puis, depuis le dossier du TP :

    spark-submit --master "local[2]" mon_script.py

In [ ]:
spark.stop()